# Import Library

In [4]:
!pip install rouge_score datasets evaluate

In [5]:
import json
import random
import torch
import evaluate
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
from transformers import BartTokenizer, BartForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments

# Custom Dataset

In [6]:
path_data = '/content/drive/Shareddrives/My Study Buddy/Pembuatan AI/Dataset/file_summarize_clean.csv'
df = pd.read_csv(path_data)
df

,Unnamed: 0,text,summary,source,text_cleaned,summary_cleaned
0,0,"- Dokter Ryan Thamrin , yang terkenal lewat ac...",Dokter Lula Kamal yang merupakan selebriti sek...,indosum,"- Dokter Ryan Thamrin , yang terkenal lewat ac...",Dokter Lula Kamal yang merupakan selebriti sek...
1,1,handset ini merupakan anggota keluarga ZenFone...,sama dibekali setup kamera ganda di depan . Me...,indosum,handset ini merupakan anggota keluarga ZenFone...,Asus memperkenalkan ZenFone generasi keempat d...
2,2,- Dinas Pariwisata Provinsi Bengkulu kembali m...,10 November 2017 yang lalu . Kegiatan yang ber...,indosum,- Dinas Pariwisata Provinsi Bengkulu kembali m...,Dinas Pariwisata Provinsi Bengkulu kembali men...
3,3,Indonesia Corruption Watch ICW meminta Komisi ...,"KTP , Johannes Marliem dan menjelaskan kepada ...",indosum,Indonesia Corruption Watch ICW meminta Komisi ...,Indonesia Corruption Watch ICW meminta Komisi ...
4,4,Presiden Joko Widodo Jokowi memimpin upacara p...,bagi sepeda kepada tamu undangan yang mengenak...,indosum,Presiden Joko Widodo Jokowi memimpin upacara p...,Jokowi memimpin upacara penurunan bendera . Us...
...,...,...,...,...,...,...
11511,11559,teror terbesar dalam sejarah Australia dilangs...,Sedikitnya seorang pria bersenjata menyandera ...,xlsum,teror terbesar dalam sejarah Australia dilangs...,Sedikitnya seorang pria bersenjata menyandera ...
11512,11560,"sama dengan Boediono selaku Gubernur BI, Miran...","Mantan Deputi Gubernur Bank Indonesia, Budi Mu...",xlsum,"sama dengan Boediono selaku Gubernur BI, Miran...","Mantan Deputi Gubernur Bank Indonesia, Budi Mu..."
11513,11561,Yahudi tidak dapat hidup. Acara di Berlin terj...,"Yahudi, sementara terjadi peningkatan serangan...",xlsum,Yahudi tidak dapat hidup. Acara di Berlin terj...,Kanselir Jerman Angela Merkel mulai berbicara ...
11514,11562,lagi Seydou Keita menyelamatkan AS Roma dari k...,1 dalam laga pertama babak 16 besar Liga Eropa...,xlsum,lagi Seydou Keita menyelamatkan AS Roma dari k...,Penampilan AS Roma tak kunjung membaik setelah...


In [7]:
# convert text ke token model
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

# load pretrained model
model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [8]:
class SummaryDataset(Dataset):
    def __init__(self, tokenizer, data, max_source_length=512, max_target_length=150):
        self.tokenizer = tokenizer
        self.data = data
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        text = item['text_cleaned']
        summary = item['summary_cleaned']

        source = self.tokenizer(
            text,
            max_length=self.max_source_length,
            truncation=True
        )

        target = self.tokenizer(
            summary,
            max_length=self.max_target_length,
            truncation=True
        )

        return {
            'input_ids': source['input_ids'],
            'attention_mask': source['attention_mask'],
            'labels': target['input_ids']
        }

In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [10]:
train_dataset = SummaryDataset(tokenizer, train_df)
test_dataset = SummaryDataset(tokenizer, test_df)

In [11]:
train_dataset[0]

{'input_ids': [0,
  29,
  16231,
  1168,
  1423,
  1097,
  7252,
  783,
  385,
  22966,
  337,
  1021,
  459,
  298,
  7821,
  991,
  415,
  967,
  17144,
  4,
  208,
  2961,
  449,
  3644,
  181,
  3314,
  2399,
  811,
  16207,
  18217,
  11,
  118,
  26951,
  677,
  2269,
  267,
  523,
  4970,
  260,
  6,
  449,
  1322,
  2133,
  2269,
  1097,
  29183,
  475,
  677,
  282,
  12837,
  19676,
  895,
  385,
  967,
  594,
  895,
  3371,
  4,
  6579,
  11,
  118,
  449,
  625,
  1097,
  604,
  219,
  922,
  405,
  11334,
  3298,
  118,
  9198,
  2348,
  1423,
  1097,
  19267,
  424,
  955,
  13163,
  7372,
  8318,
  2543,
  271,
  11,
  118,
  385,
  7375,
  3494,
  354,
  853,
  604,
  710,
  1182,
  1236,
  225,
  354,
  16207,
  18217,
  20435,
  14548,
  6629,
  1906,
  11334,
  7587,
  1350,
  26012,
  43983,
  1906,
  895,
  7670,
  438,
  9063,
  16207,
  18217,
  12,
  29,
  16231,
  1168,
  1423,
  1097,
  449,
  710,
  1097,
  385,
  22966,
  337,
  385,
  1512,
  7670,
  3914,


# Fine tuning model

In [12]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/Shareddrives/My Study Buddy/Pembuatan AI/Notebooks/results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    report_to='none',
    # --- TAMBAHAN KRUSIAL ---
    predict_with_generate=True,
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator # Masukkan collator-nya di sini
)

trainer.train()

Step,Training Loss
10,3.108643
20,3.058888
30,2.972205
40,2.560273
50,2.817615
60,2.608229
70,2.299774
80,2.452646
90,2.522239
100,2.241292


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2304, training_loss=1.6943884421553876, metrics={'train_runtime': 974.6529, 'train_samples_per_second': 18.903, 'train_steps_per_second': 2.364, 'total_flos': 5616677892833280.0, 'train_loss': 1.6943884421553876, 'epoch': 2.0})

# Evaluasi Model

In [14]:
rouge_metric = evaluate.load('rouge')

def generate_summaries_batch(model, tokenizer, dataset, data_collator, batch_size=8):
    model.eval()
    summaries = []

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
        shuffle=False
    )

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Membuat Ringkasan"):
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)

            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=150,
                num_beams=4,
                early_stopping=True,
                decoder_start_token_id=tokenizer.bos_token_id
            )

            # decode
            batch_summaries = [tokenizer.decode(ids, skip_special_tokens=True) for ids in outputs]
            summaries.extend(batch_summaries)

    return summaries


In [15]:
actual_summaries = test_df['summary_cleaned'].tolist()

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
print("Mulai proses inferensi batch...")
generated_summaries = generate_summaries_batch(
    model,
    tokenizer,
    test_dataset,
    data_collator,
    batch_size=8
)

print("\nMenghitung Skor ROUGE...")
rouge_score = rouge_metric.compute(predictions=generated_summaries, references=actual_summaries)


Mulai proses inferensi batch...


Membuat Ringkasan: 100%|██████████| 288/288 [18:16<00:00,  3.81s/it]



Menghitung Skor ROUGE...


In [16]:
print(rouge_score)

{'rouge1': np.float64(0.37202191445122795), 'rouge2': np.float64(0.26754098673539595), 'rougeL': np.float64(0.3314201925903299), 'rougeLsum': np.float64(0.33199098554771417)}


In [17]:
print("=== CEK ISI PREDIKSI (3 Data Pertama) ===")
for i, teks in enumerate(generated_summaries[:3]):
    print(f"Prediksi {i+1}: '{teks}'")

print("\n=== CEK ISI REFERENSI (3 Data Pertama) ===")
for i, teks in enumerate(actual_summaries[:3]):
    print(f"Referensi {i+1}: '{teks}'")

=== CEK ISI PREDIKSI (3 Data Pertama) ===
Prediksi 1: 'Juara dunia Grand Prix sepeda motor 500cc 1993 , Kevin Schwantz , menganggap Maverick Vinales punya tantangan besar dalam usahanya merebut gelar juara di MotoGP 2017 . Tantangan utama yang harus dihadapi vinales adalah beradu taktik melawan Valentino Rossi dan Marquez . Vinalees menjadi salah satu kandidat juara musim ini . Hingga paruh musim 2017 , mantan pebalap Suzuki itu mampu meraih tiga kemenangan dan berada di posisi kedua klasemen sementara hing'
Prediksi 2: 'Pengakuan tersebut datang ketika Nolan tengah diwawancarai oleh saudaranya , Jonathan , untuk sebuah materi promo Dunkirk . Nolan mengakui , ketika melakukan riset untuk film Tersebut , ia amat tertarik pada detail kisah evakuasi tentara Sekutu itu sehingga sangat percaya diri dapat menggarapnya tanpa naskah . Namun justru itulah yang ia usulkan kepada Crowley dan Thomas .'
Prediksi 3: 'Pelatih legendaris AC Milan , Fabio Capello 70 , mengaku ingin memeluk Gianluigi Do

In [18]:
print("\n=== HASIL EVALUASI ROUGE ===")
print(f"ROUGE-1 : {rouge_score['rouge1'] * 100:.2f}%")
print(f"ROUGE-2 : {rouge_score['rouge2'] * 100:.2f}%")
print(f"ROUGE-L : {rouge_score['rougeL'] * 100:.2f}%")


=== HASIL EVALUASI ROUGE ===
ROUGE-1 : 37.20%
ROUGE-2 : 26.75%
ROUGE-L : 33.14%


# Save Model

In [19]:
import os

folder_penyimpanan = "/content/drive/Shareddrives/My Study Buddy/Pembuatan AI/Notebooks/model2"

os.makedirs(folder_penyimpanan, exist_ok=True)

print("Sedang mengekspor model ke Google Drive...")
model.save_pretrained(folder_penyimpanan)
tokenizer.save_pretrained(folder_penyimpanan)

print(f"🎉 Selesai! Model berhasil disimpan di:\n{folder_penyimpanan}")

Sedang mengekspor model ke Google Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Selesai! Model berhasil diselamatkan secara permanen di:
/content/drive/Shareddrives/My Study Buddy/Pembuatan AI/Notebooks/model2


# Export Model
Model disimpan ke dalam format ONNX

In [3]:
# install dependencies
!pip install optimum[onnx]
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.7 MB/s eta 0:00:00


In [4]:
!optimum-cli export onnx \
  --model "/content/drive/Shareddrives/My Study Buddy/Pembuatan AI/Notebooks/model2" \
  --task text2text-generation \
  "/content/drive/Shareddrives/My Study Buddy/Pembuatan AI/Productions/Model2"

2026-05-23 13:23:25.456160: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of